# Đề tài 3: Medical Image Segmentation với UNet
# Phân đoạn Polyp từ ảnh nội soi (Kvasir-SEG Dataset)

**Mục tiêu:** Xây dựng mô hình UNet11 (dựa trên VGG-11 encoder) để tự động phân đoạn (segment) các khối polyp trong ảnh nội soi đường tiêu hóa.

**Họ và tên:** [Tên của bạn] - [MSSV]

---


## Cài đặt thư viện cần thiết


In [ ]:
# Tải bộ dataset Kvasir-SEG (nếu chạy trên Kaggle/Colab)
# !wget https://datasets.simula.no/downloads/kvasir-seg.zip
# !unzip -q kvasir-seg.zip -d data/
# Lưu ý: Sửa đường dẫn ROOT_DIR bên dưới trỏ tới thư mục chứa dữ liệu đã giải nén.


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.model_selection import train_test_split
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


---
## Bài tập 1: Data Visualization

Đọc và hiển thị một vài cặp ảnh gốc (Image) và ảnh nhãn (Mask).


In [ ]:
ROOT_DIR = "data/Kvasir-SEG" # Đổi đường dẫn này nếu cần
IMAGE_DIR = os.path.join(ROOT_DIR, "images")
MASK_DIR = os.path.join(ROOT_DIR, "masks")

# Lấy danh sách tên file
if os.path.exists(IMAGE_DIR):
    image_files = sorted(os.listdir(IMAGE_DIR))
    mask_files = sorted(os.listdir(MASK_DIR))
    print(f"Tìm thấy {len(image_files)} ảnh.")
else:
    print(f"Chưa tìm thấy thư mục {IMAGE_DIR}. Vui lòng tải dữ liệu trước.")
    image_files, mask_files = [], []


In [ ]:
# --- EXERCISE 1 ---
def visualize_samples(image_dir, mask_dir, filenames, num_samples=3):
    """Hàm hiển thị cặp ảnh và mask"""
    if not filenames: return
    
    fig, axes = plt.subplots(num_samples, 2, figsize=(10, 4 * num_samples))
    for i in range(num_samples):
        # Đọc ảnh
        img_path = os.path.join(image_dir, filenames[i])
        mask_path = os.path.join(mask_dir, filenames[i])
        
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        
        axes[i, 0].imshow(img)
        axes[i, 0].set_title(f"Original Image {i+1}")
        axes[i, 0].axis("off")
        
        axes[i, 1].imshow(mask, cmap="gray")
        axes[i, 1].set_title(f"Ground Truth Mask {i+1}")
        axes[i, 1].axis("off")
        
    plt.tight_layout()
    plt.show()

visualize_samples(IMAGE_DIR, MASK_DIR, image_files, num_samples=3)


---
## Bài tập 2: Dataset & Data Transform

Tạo class Dataset và thực hiện tiền xử lý:
- Resize ảnh về 224x224
- Chuẩn hóa ảnh (Normalize) theo ImageNet
- Nhị phân hóa (Binarize) ảnh mask


In [ ]:
# --- EXERCISE 2 ---
class PolypDataset(Dataset):
    def __init__(self, image_dir, mask_dir, filenames):
        self.image_dir = image_dir
        self.mask_dir = mask_dir
        self.filenames = filenames
        self.size = (224, 224)
        
        # Transform cho ảnh gốc (Resize -> ToTensor -> Normalize)
        self.img_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(self.size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], 
                                 std=[0.229, 0.224, 0.225])
        ])
        
        # Transform cho mask (Resize -> ToTensor)
        self.mask_transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize(self.size),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        img_name = self.filenames[idx]
        
        # Đọc và biến đổi ảnh
        img_path = os.path.join(self.image_dir, img_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = self.img_transform(img)
        
        # Đọc và biến đổi mask
        mask_path = os.path.join(self.mask_dir, img_name)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = self.mask_transform(mask)
        
        # Binarize mask: Đưa giá trị về {0, 1}
        mask = torch.where(mask > 0.5, torch.tensor(1.0), torch.tensor(0.0))
        
        return img, mask


In [ ]:
if image_files:
    # Chia train/val/test tỉ lệ 60/20/20
    train_files, temp_files = train_test_split(image_files, test_size=0.4, random_state=42)
    val_files, test_files = train_test_split(temp_files, test_size=0.5, random_state=42)
    
    train_dataset = PolypDataset(IMAGE_DIR, MASK_DIR, train_files)
    val_dataset = PolypDataset(IMAGE_DIR, MASK_DIR, val_files)
    test_dataset = PolypDataset(IMAGE_DIR, MASK_DIR, test_files)
    
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False)
    
    print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")


---
## Bài tập 3: Xây dựng mô hình UNet11

Sử dụng VGG-11 (pre-trained) làm phần Encoder. Mạng UNet sẽ có kiến trúc hình chữ U với các kết nối ngang (skip connections).


In [ ]:
class DecoderBlock(nn.Module):
    """Khối Decoder cơ bản: ConvTranspose2d -> ReLU -> Conv2d -> ReLU"""
    def __init__(self, in_channels, middle_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True)
        )
    def forward(self, x):
        return self.block(x)

# --- EXERCISE 3 ---
class UNet11(nn.Module):
    def __init__(self, num_classes=1):
        super().__init__()
        # Tải VGG-11 pre-trained (sử dụng features)
        vgg = models.vgg11(pretrained=True).features
        
        # Encoder (Dùng các lớp của VGG11)
        self.enc1 = vgg[0:3]   # 64 channels, kích thước không đổi
        self.enc2 = vgg[3:6]   # 128 channels, giảm 1/2
        self.enc3 = vgg[6:11]  # 256 channels, giảm 1/4
        self.enc4 = vgg[11:16] # 512 channels, giảm 1/8
        self.enc5 = vgg[16:21] # 512 channels, giảm 1/16
        
        # Lớp trung gian (Center)
        self.center = DecoderBlock(512, 512, 256)
        
        # Decoder (Có nhận thêm skip connection từ Encoder)
        self.dec5 = DecoderBlock(512 + 256, 512, 256)
        self.dec4 = DecoderBlock(256 + 256, 256, 128)
        self.dec3 = DecoderBlock(128 + 128, 128, 64)
        self.dec2 = DecoderBlock(64 + 64, 64, 32)
        
        self.dec1 = nn.Sequential(
            nn.Conv2d(32 + 3, 32, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        
        self.final = nn.Conv2d(32, num_classes, kernel_size=1)

    def forward(self, x):
        # Xuôi qua Encoder (lưu lại features để làm skip connection)
        e1 = self.enc1(x)
        e2 = self.enc2(e1)
        e3 = self.enc3(e2)
        e4 = self.enc4(e3)
        e5 = self.enc5(e4)
        
        # Center
        c = self.center(e5)
        
        # Decoder kết hợp với Skip Connections
        d5 = self.dec5(torch.cat([c, e4], dim=1))
        d4 = self.dec4(torch.cat([d5, e3], dim=1))
        d3 = self.dec3(torch.cat([d4, e2], dim=1))
        d2 = self.dec2(torch.cat([d3, e1], dim=1))
        
        d1 = self.dec1(torch.cat([d2, x], dim=1))
        
        out = self.final(d1)
        return out

model = UNet11().to(device)
print(f"Tổng số tham số: {sum(p.numel() for p in model.parameters()):,}")


---
## Bài tập 4: Đánh giá mô hình (IoU)

Cài đặt hàm tính Intersection over Union (IoU) - còn gọi là Jaccard Index.
Công thức: `IoU = Intersection / Union`


In [ ]:
# --- EXERCISE 4 ---
def compute_iou(y_pred, y_true):
    """
    Tính IoU.
    y_pred: tensor (đã qua sigmoid), shape [B, 1, H, W]
    y_true: tensor (nhãn), shape [B, 1, H, W]
    """
    # Chuyển prediction thành nhị phân với ngưỡng 0.5
    y_pred_bin = (y_pred > 0.5).float()
    
    intersection = (y_pred_bin * y_true).sum(dim=(2, 3))
    union = y_pred_bin.sum(dim=(2, 3)) + y_true.sum(dim=(2, 3)) - intersection
    
    # Tránh chia cho 0
    iou = (intersection + 1e-6) / (union + 1e-6)
    return iou.mean().item()


---
## Huấn luyện mô hình (Training)


In [ ]:
# Khởi tạo hàm loss và optimizer
# BCEWithLogitsLoss đã bao gồm sẵn hàm Sigmoid
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=4e-5, weight_decay=1e-4)

EPOCHS = 20

def train_one_epoch(model, dataloader, optimizer, criterion):
    model.train()
    total_loss = 0
    total_iou = 0
    
    for inputs, targets in tqdm(dataloader, desc="Training"):
        inputs, targets = inputs.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_iou += compute_iou(torch.sigmoid(outputs), targets)
        
    return total_loss / len(dataloader), total_iou / len(dataloader)

def validate(model, dataloader, criterion):
    model.eval()
    total_loss = 0
    total_iou = 0
    
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc="Validation"):
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            
            loss = criterion(outputs, targets)
            total_loss += loss.item()
            total_iou += compute_iou(torch.sigmoid(outputs), targets)
            
    return total_loss / len(dataloader), total_iou / len(dataloader)


In [ ]:
# --- BỎ COMMENT ĐOẠN DƯỚI ĐỂ TRAIN ---
"""
train_losses, val_losses = [], []
train_ious, val_ious = [], []

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss, train_iou = train_one_epoch(model, train_loader, optimizer, criterion)
    val_loss, val_iou = validate(model, val_loader, criterion)
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_ious.append(train_iou)
    val_ious.append(val_iou)
    
    print(f"Train Loss: {train_loss:.4f} | Train IoU: {train_iou:.4f}")
    print(f"Val Loss:   {val_loss:.4f} | Val IoU:   {val_iou:.4f}")

# Vẽ biểu đồ Loss và IoU
plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.title('Loss over Epochs')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(train_ious, label='Train IoU')
plt.plot(val_ious, label='Val IoU')
plt.title('IoU over Epochs')
plt.legend()
plt.show()

# Đánh giá trên tập Test
test_loss, test_iou = validate(model, test_loader, criterion)
print(f"\n--- KẾT QUẢ TRÊN TẬP TEST ---")
print(f"Test Loss: {test_loss:.4f} | Test IoU: {test_iou:.4f}")
"""
print("Mã huấn luyện đã sẵn sàng. Hãy tải dataset và chạy!")


---
## Kiểm tra kết quả (Inference)

Hàm để dự đoán và vẽ ảnh kết quả của mô hình sau khi train.


In [ ]:
def visualize_prediction(model, dataset, idx):
    if not image_files: return
    model.eval()
    
    img_tensor, mask_tensor = dataset[idx]
    
    # Dự đoán
    with torch.no_grad():
        out = model(img_tensor.unsqueeze(0).to(device))
        pred_mask = torch.sigmoid(out).squeeze().cpu().numpy()
        pred_bin = (pred_mask > 0.5).astype(np.float32)
        
    # Chuẩn bị ảnh gốc để vẽ
    img_show = img_tensor.permute(1, 2, 0).numpy()
    # Denormalize
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    img_show = std * img_show + mean
    img_show = np.clip(img_show, 0, 1)
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_show)
    axes[0].set_title("Input Image")
    axes[0].axis('off')
    
    axes[1].imshow(mask_tensor.squeeze(), cmap='gray')
    axes[1].set_title("Ground Truth Mask")
    axes[1].axis('off')
    
    axes[2].imshow(pred_bin, cmap='gray')
    axes[2].set_title("Predicted Mask")
    axes[2].axis('off')
    
    plt.show()

# --- BỎ COMMENT DÒNG DƯỚI ĐỂ VẼ ẢNH DỰ ĐOÁN TỪ TẬP TEST ---
# visualize_prediction(model, test_dataset, 0)
# visualize_prediction(model, test_dataset, 1)


---
## Đánh giá lâm sàng (Clinical Evaluation)

Trong thực tế, mô hình có thể dự đoán sai (Dương tính giả - False Positive) ở một số trường hợp. Ví dụ: AI nhận diện một nếp gấp đại tràng bình thường là polyp do nếp gấp này **nhô lên và bị ánh đèn nội soi chiếu vào tạo độ bóng (Specular highlight)**. Cấu trúc nhô cao kèm độ bóng sáng rất giống với đặc điểm hình ảnh của polyp mà AI đã học.

Đây là hạn chế của việc phân tích trên ảnh tĩnh 2D. Trong lâm sàng, bác sĩ có thể bơm rửa, quan sát từ nhiều góc độ qua video, hoặc đổi chế độ ánh sáng để xác nhận. Tuy nhiên, trong vai trò công cụ sàng lọc, việc AI có độ nhạy cao (thà "bắt nhầm" để bác sĩ kiểm tra lại còn hơn "bỏ sót" tổn thương) là một đặc tính rất cần thiết.

